In [1]:
import os
from pathlib import Path
import json
import torch
from nemo.collections.asr.models import EncDecSpeakerLabelModel
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

c:\Users\Somlab\miniforge3\envs\bb_voice_transcription\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-07-18 18:09:42 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
W0718 18:09:42.267000 5956 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [33]:
# Identify the sound file and transcript
tld = Path(r"C:\Users\Somlab\Downloads")
fname = "audio1365983309"
audio_path = fname + ".16k.wav"
transcript_path = fname + ".transcript.json"

with open(tld / transcript_path, 'r', encoding='utf8') as f: 
    transcript = json.load(f)

print(transcript)

[{'start': 0.0, 'end': 2.48, 'speaker': 'speaker_3', 'text': 'and perfect.'}, {'start': 3.3, 'end': 5.14, 'speaker': 'speaker_0', 'text': 'Okay, so for this first section,'}, {'start': 5.34, 'end': 7.32, 'speaker': 'speaker_0', 'text': "I'm just gonna be asking about, you know,"}, {'start': 7.42, 'end': 9.9, 'speaker': 'speaker_0', 'text': 'how life has been since your testing ended'}, {'start': 9.9, 'end': 11.94, 'speaker': 'speaker_0', 'text': 'and your thoughts and feelings about that.'}, {'start': 12.9, 'end': 15.6, 'speaker': 'speaker_0', 'text': 'Thinking about your breast sensation now,'}, {'start': 16.3, 'end': 17.96, 'speaker': 'speaker_0', 'text': 'how do you think it compares'}, {'start': 17.96, 'end': 20.02, 'speaker': 'speaker_0', 'text': 'to what you felt during testing?'}, {'start': 22.36, 'end': 25.52, 'speaker': 'speaker_3', 'text': 'I feel as the time is progressing,'}, {'start': 25.96, 'end': 29.08, 'speaker': 'speaker_3', 'text': "I feel that I'm getting more sensat

In [ ]:
# Get segments split by predicted speaker
speaker = transcript[0]['speaker']
num_segments = len(transcript)
speaker_start_times, speaker_stop_times, speakers = [0], [], [speaker]
for i, t in enumerate(transcript):
    if t['speaker'] != speaker:
        speaker_start_times.append(t['start'])
        if i == 0 or i == num_segments:
            speaker_stop_times.append(t['end'])
        else:
            speaker_stop_times.append(transcript[i-1]['end'])
        speaker = t['speaker']
        speakers.append(speaker)

# Add the final timestamp
speaker_stop_times.append(transcript[-1]['end'])

# Get an encoding model for feature extraction
enc_model = EncDecSpeakerLabelModel.from_pretrained(model_name="titanet_small").eval()
device = next(enc_model.parameters()).device
# load the whole audio file
sample_frequency = 16000
audio, _ = librosa.load(tld / audio_path, sr=sample_frequency)


['unknown', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_10', 'speaker_11', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'speaker_21', 'speaker_20', 'unknown', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_31', 'speaker_30', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_41', 'speaker_40', 'speaker_51', 'speaker_50', 'speaker_51', 'speaker_50', 'speaker_51

[NeMo W 2026-07-18 18:11:04 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels:
    - '100'
    - '1001'
    - 100304-f
    - 100396-m
    - 1004-f
    - 100568-f
    - '1006'
    - 1006-m
    - 1008-f
    - 1009-f
    - 100905-m
    - 100909-f
    - 100910-f
    - 100912-f
    - 100914-f
    - 100915-f
    - 100921-m
    - 100922-m
    - 100924-f
    - 100926-f
    - 100931-m
    - 100934-m
    - 100935-f
    - 100937-f
    - 100938-m
    - 100939-m
    - 100942-f
    - 100943-m
    - 100948-f
    - 100952-m
    - 100953-f
    - 100954-m
    - 100956-f
    - 100957-f
    - 100959-m
    - 100960-f
    - 100961-m
    - 100962-m
    - 100963-m
    - 100964-f
    - 100966-f
    - 100975-f
    - 100978-f
    - 100980-f
 

[NeMo I 2026-07-18 18:11:04 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from C:\Users\Somlab\.cache\torch\NeMo\NeMo_2.7.3\titanet-s\908e9576cf7dd7420e75f73ceb0b72e1\titanet-s.nemo.


In [ ]:
# Manually compute embeddings
embeddings = []
for (start, stop) in zip(speaker_start_times, speaker_stop_times):
    # Get segment indices for audio file
    start_idx = int(start * sample_frequency)
    stop_idx = int(stop * sample_frequency)
    audio_tensor = torch.tensor(audio[start_idx:stop_idx], dtype=torch.float32).unsqueeze(0).to(device)
    audio_len = torch.tensor([audio_tensor.shape[1]], dtype=torch.int32).to(device)

    # Get the embedding from the encoding model
    with torch.no_grad():
        _, embedding = enc_model.forward(input_signal=audio_tensor, input_signal_length=audio_len)

    # Normalize and return to CPU
    embedding = embedding.squeeze(0).cpu().numpy()
    embeddings.append(embedding / np.linalg.norm(embedding))

# Compute similarity between each pair of embeddings
similarity_mat = np.zeros((len(embeddings), len(embeddings)))
for i in range(len(embeddings)):
    for j in range(len(embeddings)):
        similarity_mat[i,j] = np.dot(embeddings[i], embeddings[j]) / ((np.dot(embeddings[i], embeddings[i]) * np.dot(embeddings[j], embeddings[j])) ** 0.5)

# Convert to distance matrix for precomputed clustering
distance_mat = 1-similarity_mat

# Cluster to get harmonized speaker labels
feature_clusterer = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average', distance_threshold=0.75)
feature_labels = feature_clusterer.fit_predict(distance_mat)

In [29]:
for t in transcript:
    t['speaker'] = f"speaker_{feature_labels[speakers.index(t['speaker'])]}"

transcript


[{'start': 0.0, 'end': 2.48, 'speaker': 'speaker_1', 'text': 'and perfect.'},
 {'start': 3.3,
  'end': 5.14,
  'speaker': 'speaker_0',
  'text': 'Okay, so for this first section,'},
 {'start': 5.34,
  'end': 7.32,
  'speaker': 'speaker_0',
  'text': "I'm just gonna be asking about, you know,"},
 {'start': 7.42,
  'end': 9.9,
  'speaker': 'speaker_0',
  'text': 'how life has been since your testing ended'},
 {'start': 9.9,
  'end': 11.94,
  'speaker': 'speaker_0',
  'text': 'and your thoughts and feelings about that.'},
 {'start': 12.9,
  'end': 15.6,
  'speaker': 'speaker_0',
  'text': 'Thinking about your breast sensation now,'},
 {'start': 16.3,
  'end': 17.96,
  'speaker': 'speaker_0',
  'text': 'how do you think it compares'},
 {'start': 17.96,
  'end': 20.02,
  'speaker': 'speaker_0',
  'text': 'to what you felt during testing?'},
 {'start': 22.36,
  'end': 25.52,
  'speaker': 'speaker_3',
  'text': 'I feel as the time is progressing,'},
 {'start': 25.96,
  'end': 29.08,
  'speake

In [37]:
speaker_id = [int(t['speaker'].split('_')[-1]) for t in transcript]
speaker_list = list(set(speaker_id))
speaker_list

[0, 3]